# Risk Scoring Model — Baseline Evaluation

Evaluate the risk scoring XGBoost regression model (`risk-scoring-xgboost-v1`).

**Model:** `gs://i4g-ml-data/models/risk-scoring-xgboost-v1-20260328-0447`  
**Endpoint:** `/predict/risk-score` on `ml-serving`  
**Metrics:** MSE, MAE, RMSE, Spearman ρ, score distribution, residual analysis

**Usage:** Set `MODEL_ARTIFACT_URI` below, then run all cells.

In [ ]:
from __future__ import annotations

import json
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb

## Configuration

In [ ]:
MODEL_ARTIFACT_URI = "gs://i4g-ml-data/models/risk-scoring-xgboost-v1-20260328-0447"

# Evaluation dataset (JSONL with features + severity/risk_score ground truth)
EVAL_DATA_PATH = "../../data/local-training/eval.jsonl"

# Where to save baseline snapshot
BASELINE_OUTPUT_PATH = "../../data/eval/risk_baseline_result.json"

## Load Model Artifacts from GCS

In [ ]:
from google.cloud import storage


def download_model(artifact_uri: str, dest: Path) -> None:
    """Download model artifacts from GCS to a local directory."""
    uri = artifact_uri.replace("gs://", "")
    bucket_name = uri.split("/")[0]
    prefix = "/".join(uri.split("/")[1:])
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    for blob in bucket.list_blobs(prefix=prefix):
        rel = blob.name[len(prefix):].lstrip("/")
        if not rel:
            continue
        local_path = dest / rel
        local_path.parent.mkdir(parents=True, exist_ok=True)
        blob.download_to_filename(str(local_path))
        print(f"  ↓ {rel}")


model_dir = Path(tempfile.mkdtemp(prefix="risk_eval_"))
print(f"Downloading model to {model_dir}")
download_model(MODEL_ARTIFACT_URI, model_dir)

# Load XGBoost model
booster = xgb.Booster()
booster.load_model(str(model_dir / "xgboost_model.json"))

# Load feature columns
feature_cols = json.loads((model_dir / "feature_cols.json").read_text())
print(f"\nFeature columns ({len(feature_cols)}): {feature_cols}")

# Load training metrics
metrics_path = model_dir / "metrics.json"
if metrics_path.exists():
    training_metrics = json.loads(metrics_path.read_text())
    print(f"Training metrics: {training_metrics}")
else:
    training_metrics = None
    print("No training metrics found")

## Load Evaluation Data

In [ ]:
records: list[dict] = []
with open(EVAL_DATA_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f"Loaded {len(records)} evaluation samples")

# Extract features and ground truth
# Ground truth: "severity" field (0.0–1.0) or "risk_score" if available
rows = []
for rec in records:
    feat = rec.get("features", {})
    # Ground truth comes from severity or risk_score label
    gt = float(rec.get("severity", rec.get("risk_score", rec.get("labels", {}).get("severity", 0.5))))
    rows.append({"case_id": rec["case_id"], "ground_truth": gt, **feat})

df = pd.DataFrame(rows)
print(f"Columns: {list(df.columns)}")
df.head()

## Generate Predictions

In [ ]:
# Build feature matrix using the model's expected column order
X = df[feature_cols].fillna(0).values.astype(np.float32)
dmat = xgb.DMatrix(X, feature_names=feature_cols)
raw_preds = booster.predict(dmat)

# Clamp to [0, 1]
predictions = np.clip(raw_preds, 0.0, 1.0)
df["predicted_risk"] = predictions

print(f"Predictions: min={predictions.min():.4f}, max={predictions.max():.4f}, mean={predictions.mean():.4f}")
gt = df['ground_truth']
print(f"Ground truth: min={gt.min():.4f}, max={gt.max():.4f}, mean={gt.mean():.4f}")


## Regression Metrics

In [ ]:
from ml.training.evaluation import evaluate_regression

result = evaluate_regression(
    predictions=df["predicted_risk"].tolist(),
    ground_truth=df["ground_truth"].tolist(),
)

print(result.summary())
print()

metrics_df = pd.DataFrame([{
    "Metric": "MSE", "Value": f"{result.mse:.4f}",
}, {
    "Metric": "MAE", "Value": f"{result.mae:.4f}",
}, {
    "Metric": "RMSE", "Value": f"{result.rmse:.4f}",
}, {
    "Metric": "Spearman ρ", "Value": f"{result.spearman_rho:.4f}",
}, {
    "Metric": "Samples", "Value": str(result.total_samples),
}])
metrics_df

## Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Predicted vs ground truth histogram
axes[0].hist(df["ground_truth"], bins=20, alpha=0.6, label="Ground Truth", color="steelblue")
axes[0].hist(df["predicted_risk"], bins=20, alpha=0.6, label="Predicted", color="coral")
axes[0].set_xlabel("Risk Score")
axes[0].set_ylabel("Count")
axes[0].set_title("Score Distribution")
axes[0].legend()

# Predicted vs ground truth scatter
axes[1].scatter(df["ground_truth"], df["predicted_risk"], alpha=0.6, s=30, color="steelblue")
axes[1].plot([0, 1], [0, 1], "r--", label="Perfect")
axes[1].set_xlabel("Ground Truth")
axes[1].set_ylabel("Predicted")
axes[1].set_title(f"Predicted vs Ground Truth (ρ={result.spearman_rho:.3f})")
axes[1].set_xlim(-0.05, 1.05)
axes[1].set_ylim(-0.05, 1.05)
axes[1].legend()

plt.tight_layout()
plt.show()

## Residual Analysis

In [ ]:
df["residual"] = df["predicted_risk"] - df["ground_truth"]
df["abs_error"] = df["residual"].abs()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Residual distribution
axes[0].hist(df["residual"], bins=20, color="steelblue", edgecolor="white")
axes[0].axvline(x=0, color="red", linestyle="--")
axes[0].set_xlabel("Residual (Predicted − Truth)")
axes[0].set_ylabel("Count")
axes[0].set_title(f"Residual Distribution (mean={df['residual'].mean():.4f})")

# Residual vs predicted
axes[1].scatter(df["predicted_risk"], df["residual"], alpha=0.6, s=30, color="steelblue")
axes[1].axhline(y=0, color="red", linestyle="--")
axes[1].set_xlabel("Predicted Risk Score")
axes[1].set_ylabel("Residual")
axes[1].set_title("Residuals vs Predicted")

plt.tight_layout()
plt.show()

print("\nTop 5 highest-error samples:")
df.nlargest(5, "abs_error")[["case_id", "ground_truth", "predicted_risk", "residual"]].to_string(index=False)

## Feature Importance

In [ ]:
importance = booster.get_score(importance_type="gain")
imp_df = pd.DataFrame([
    {"Feature": k, "Gain": v} for k, v in importance.items()
]).sort_values("Gain", ascending=True)

fig, ax = plt.subplots(figsize=(8, max(3, len(imp_df) * 0.4)))
ax.barh(imp_df["Feature"], imp_df["Gain"], color="steelblue")
ax.set_xlabel("Gain")
ax.set_title("Feature Importance (Gain)")
plt.tight_layout()
plt.show()

imp_df.sort_values("Gain", ascending=False)

## Baseline Comparison

Load a saved baseline and compare. If no baseline exists, save this run as the baseline.

In [ ]:
baseline_path = Path(BASELINE_OUTPUT_PATH)

current = {
    "model_artifact_uri": MODEL_ARTIFACT_URI,
    "mse": result.mse,
    "mae": result.mae,
    "rmse": result.rmse,
    "spearman_rho": result.spearman_rho,
    "total_samples": result.total_samples,
}

if baseline_path.exists():
    baseline = json.loads(baseline_path.read_text())
    print(f"Baseline model: {baseline.get('model_artifact_uri', 'unknown')}")
    print()
    comparison = pd.DataFrame([{
        "Metric": "MSE",
        "Baseline": f"{baseline['mse']:.4f}",
        "Current": f"{current['mse']:.4f}",
        "Δ": f"{current['mse'] - baseline['mse']:+.4f}",
        "Better?": "✓" if current["mse"] <= baseline["mse"] else "✗",
    }, {
        "Metric": "MAE",
        "Baseline": f"{baseline['mae']:.4f}",
        "Current": f"{current['mae']:.4f}",
        "Δ": f"{current['mae'] - baseline['mae']:+.4f}",
        "Better?": "✓" if current["mae"] <= baseline["mae"] else "✗",
    }, {
        "Metric": "RMSE",
        "Baseline": f"{baseline['rmse']:.4f}",
        "Current": f"{current['rmse']:.4f}",
        "Δ": f"{current['rmse'] - baseline['rmse']:+.4f}",
        "Better?": "✓" if current["rmse"] <= baseline["rmse"] else "✗",
    }, {
        "Metric": "Spearman ρ",
        "Baseline": f"{baseline['spearman_rho']:.4f}",
        "Current": f"{current['spearman_rho']:.4f}",
        "Δ": f"{current['spearman_rho'] - baseline['spearman_rho']:+.4f}",
        "Better?": "✓" if current["spearman_rho"] >= baseline["spearman_rho"] else "✗",
    }])
    display(comparison)
else:
    print("No baseline found — saving current run as baseline.")
    baseline_path.parent.mkdir(parents=True, exist_ok=True)
    baseline_path.write_text(json.dumps(current, indent=2))
    print(f"Saved to {baseline_path}")

## Live Endpoint Smoke Test

Verify the deployed model returns consistent scores.

In [ ]:
import subprocess

import requests

SERVE_URL = "https://ml-serving-21208516810.us-central1.run.app"
SA = "sa-app@i4g-dev.iam.gserviceaccount.com"

token = subprocess.check_output(
    ["gcloud", "auth", "print-identity-token",
     f"--impersonate-service-account={SA}",
     f"--audiences={SERVE_URL}"],
    text=True,
).strip()

test_cases = [
    {"case_id": "eval-low", "text": "Simple inquiry about product return policy."},
    {"case_id": "eval-med", "text": "Someone asked me to wire $2,000 via Western Union to claim my prize winnings."},
    {
        "case_id": "eval-high",
        "text": (
            "Victim sent 3 BTC to wallet 1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa "
            "after romance scam on WhatsApp lasting 6 months. "
            "Account number 12345678901234."
        ),
    },
]

for tc in test_cases:
    resp = requests.post(
        f"{SERVE_URL}/predict/risk-score",
        json=tc,
        headers={"Authorization": f"Bearer {token}"},
    )
    data = resp.json()
    score = data.get("risk_score", "ERROR")
    print(f"  {tc['case_id']:>10}: risk_score={score}  (status={resp.status_code})")

## Summary

| Metric | Value |
|--------|-------|
| MSE | — |
| MAE | — |
| RMSE | — |
| Spearman ρ | — |

Fill in after running. Next steps:
- Save baseline result for future comparison
- Evaluate on larger dataset once BigQuery feature store is populated
- Compare with heuristic-only baseline (e.g., text_length thresholds)